# 🚀 ADAN Trading Bot - Colab Training V3
**Instance 3 of 5 - Independent Training Session**

This notebook runs independently. If it fails, you can restart with another version.

In [ ]:
# ============================================================================
# PHASE 1: Setup & Dependencies
# ============================================================================
import os
import sys
from pathlib import Path

# Set working directory
os.chdir('/content')
sys.path.insert(0, '/content/ADAN0')

print("📁 Working Directory:", os.getcwd())
print("🐍 Python Version:", sys.version)
print("📦 Python Path:", sys.path[:3])

In [ ]:
# ============================================================================
# PHASE 2: Clone Repository
# ============================================================================
import subprocess

if not Path('/content/ADAN0').exists():
    print("📥 Cloning ADAN repository...")
    result = subprocess.run(
        ['git', 'clone', 'https://github.com/Cabrel10/ADAN0.git'],
        cwd='/content',
        capture_output=True,
        text=True
    )
    if result.returncode == 0:
        print("✅ Repository cloned successfully")
    else:
        print(f"❌ Clone failed: {result.stderr}")
else:
    print("✅ Repository already exists")

In [ ]:
# ============================================================================
# PHASE 3: Install System Dependencies & TA-Lib
# ============================================================================
import subprocess

print("📦 Installing system dependencies...")
subprocess.run(['apt-get', 'update', '-qq'], capture_output=True)
subprocess.run([
    'apt-get', 'install', '-y', '-qq',
    'build-essential', 'wget', 'curl', 'git', 'libffi-dev', 'libssl-dev', 'python3-dev'
], capture_output=True)
print("✅ System dependencies installed")

# Check if TA-Lib is installed
try:
    import talib
    print(f"✅ TA-Lib already installed (v{talib.__version__})")
except ImportError:
    print("📥 Installing TA-Lib from source...")
    
    # Download and compile
    subprocess.run([
        'bash', '-c',
        'cd /tmp && wget -q https://prdownloads.sourceforge.net/ta-lib/ta-lib-0.4.28-src.tar.gz && '
        'tar -xzf ta-lib-0.4.28-src.tar.gz && cd ta-lib && '
        './configure --prefix=/usr --enable-shared > /dev/null 2>&1 && '
        'make -j$(nproc) > /dev/null 2>&1 && '
        'make install > /dev/null 2>&1 && '
        'ldconfig'
    ], capture_output=True)
    
    # Install Python bindings
    subprocess.run(['pip', 'install', '--no-cache-dir', 'ta-lib'], capture_output=True)
    
    import talib
    print(f"✅ TA-Lib installed successfully (v{talib.__version__})")

In [ ]:
# ============================================================================
# PHASE 4: Install Python Dependencies
# ============================================================================
import subprocess

print("📦 Installing Python dependencies...")

# Upgrade pip
subprocess.run(['pip', 'install', '--upgrade', 'pip', 'setuptools', 'wheel', '-q'], capture_output=True)

# Install requirements
requirements = [
    'torch==2.8.0',
    'numpy>=1.24.0,<2.0.0',
    'pandas>=2.0.0,<3.0.0',
    'scipy>=1.10.0,<2.0.0',
    'stable-baselines3==2.7.0',
    'gymnasium>=0.28.0,<1.0.0',
    'pandas-ta>=0.3.14b0',
    'yfinance>=0.2.32',
    'optuna>=4.5.0',
    'pyyaml>=6.0',
    'python-dotenv>=1.0.0',
    'requests>=2.31.0',
    'tensorboard>=2.14.0',
]

for req in requirements:
    print(f"  Installing {req.split('==')[0].split('>')[0]}...")
    subprocess.run(['pip', 'install', '--no-cache-dir', req], capture_output=True)

print("✅ All dependencies installed")

In [ ]:
# ============================================================================
# PHASE 5: Install ADAN Package
# ============================================================================
import subprocess
import os

os.chdir('/content/ADAN0')
print("📦 Installing ADAN package in development mode...")

result = subprocess.run(['pip', 'install', '-e', '.', '-q'], capture_output=True, text=True)
if result.returncode == 0:
    print("✅ ADAN package installed")
else:
    print(f"⚠️ Package installation warning: {result.stderr[:200]}")

In [ ]:
# ============================================================================
# PHASE 6: Verify Imports
# ============================================================================
import sys

print("🔍 Verifying critical imports...\n")

modules_to_check = [
    ('torch', 'PyTorch'),
    ('numpy', 'NumPy'),
    ('pandas', 'Pandas'),
    ('gymnasium', 'Gymnasium'),
    ('stable_baselines3', 'Stable Baselines3'),
    ('talib', 'TA-Lib'),
    ('optuna', 'Optuna'),
    ('yaml', 'PyYAML'),
]

all_ok = True
for module_name, display_name in modules_to_check:
    try:
        mod = __import__(module_name)
        version = getattr(mod, '__version__', 'unknown')
        print(f"  ✅ {display_name:<20} {version}")
    except ImportError as e:
        print(f"  ❌ {display_name:<20} FAILED")
        all_ok = False

print()
if all_ok:
    print("✅ All imports successful!")
else:
    print("❌ Some imports failed. Please check the installation.")
    sys.exit(1)

In [ ]:
# ============================================================================
# PHASE 7: Import ADAN Modules
# ============================================================================
import os
os.chdir('/content/ADAN0')
sys.path.insert(0, '/content/ADAN0')

print("📦 Importing ADAN modules...\n")

try:
    from adan_trading_bot.common.config_loader import ConfigLoader
    print("  ✅ ConfigLoader")
    
    from adan_trading_bot.environment.multi_asset_chunked_env import MultiAssetChunkedEnv
    print("  ✅ MultiAssetChunkedEnv")
    
    from adan_trading_bot.agent.ppo_agent import PPOAgent
    print("  ✅ PPOAgent")
    
    from adan_trading_bot.training.trainer import Trainer
    print("  ✅ Trainer")
    
    print("\n✅ All ADAN modules imported successfully!")
except ImportError as e:
    print(f"❌ Import error: {e}")
    sys.exit(1)

In [ ]:
# ============================================================================
# PHASE 8: Load Configuration
# ============================================================================
print("⚙️ Loading configuration...\n")

config = ConfigLoader.load_config('config/config.yaml')

print("✅ Configuration loaded")
print(f"  Capital: ${config.get('capital_initial', 20.5)}")
print(f"  Workers: {config.get('num_workers', 4)}")
print(f"  Assets: {config.get('assets', ['BTCUSDT'])}")
print(f"  Timeframes: {config.get('timeframes', ['5m', '1h', '4h'])}")

In [ ]:
# ============================================================================
# PHASE 9: Create Environment
# ============================================================================
print("🌍 Creating training environment...\n")

try:
    env = MultiAssetChunkedEnv(
        config=config,
        worker_id=0,
        mode='train'
    )
    
    obs, info = env.reset()
    print("✅ Environment created and reset")
    print(f"  Observation keys: {list(obs.keys())}")
    print(f"  Action space: {env.action_space}")
    print(f"  Observation space: {env.observation_space}")
except Exception as e:
    print(f"❌ Environment creation failed: {e}")
    import traceback
    traceback.print_exc()
    sys.exit(1)

In [ ]:
# ============================================================================
# PHASE 10: Start Training
# ============================================================================
print("🚀 Starting training...\n")
print("="*70)

import time
from datetime import datetime

start_time = time.time()
print(f"Start Time: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"Instance: V3 (Independent Session 1)")
print(f"Total Timesteps: 500,000")
print(f"Workers: 4")
print("="*70)
print()

try:
    # Create trainer
    trainer = Trainer(
        config=config,
        checkpoint_dir='checkpoints_v3',
        log_dir='logs_v3'
    )
    
    # Train
    trainer.train(
        total_timesteps=500000,
        num_workers=4,
        resume=False,
        verbose=1
    )
    
    elapsed = time.time() - start_time
    print("\n" + "="*70)
    print(f"✅ Training completed successfully!")
    print(f"Duration: {elapsed/3600:.2f} hours")
    print(f"End Time: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    print("="*70)
    
except KeyboardInterrupt:
    print("\n⚠️ Training interrupted by user")
except Exception as e:
    print(f"\n❌ Training failed: {e}")
    import traceback
    traceback.print_exc()

In [ ]:
# ============================================================================
# PHASE 11: Extract & Download Model
# ============================================================================
print("📦 Extracting trained model...\n")

from pathlib import Path
import shutil

checkpoint_dir = Path('checkpoints_v3')
if checkpoint_dir.exists():
    # Find latest checkpoint
    checkpoints = list(checkpoint_dir.glob('**/best_model.zip'))
    if checkpoints:
        latest = max(checkpoints, key=lambda p: p.stat().st_mtime)
        print(f"✅ Found checkpoint: {latest}")
        print(f"  Size: {latest.stat().st_size / 1024 / 1024:.2f} MB")
        print(f"  Modified: {datetime.fromtimestamp(latest.stat().st_mtime)}")
    else:
        print("⚠️ No checkpoint found")
else:
    print("⚠️ Checkpoint directory not found")

print("\n✅ Training session V1 complete!")